# Notebook 14.1  A late-fusion audiovisual emotion classifier and the gain from video under noise

**Goal.** Build a late-fusion classifier that combines an audio stream and a visual stream for emotion recognition, and measure how much the visual stream helps as acoustic noise increases.

**What runs here.** Everything runs with no downloads. We generate a synthetic two-stream dataset (audio and video features per utterance), train a simple per-modality classifier, fuse the two by averaging their class posteriors (late fusion), and evaluate audio-only, video-only, and audiovisual accuracy as we add noise to the audio at test time. The markdown says exactly where a real Arabic affect set would plug in. This accompanies Chapter 14.

**The point.** When the audio is clean, audio alone is enough. As the audio degrades, audio-only accuracy collapses toward chance while the fused audiovisual system stays high, because the visual stream carries information the noise destroyed. This is the chapter's core claim, made measurable. It is a teaching illustration, not a benchmark: real audiovisual affect is harder, the labels are subjective, and the streams must be aligned.

## 1. Setup

In [ ]:
import numpy as np

rng = np.random.default_rng(14)   # reproducible
EMOTIONS = ['neutral', 'happy', 'angry', 'sad']
N_CLASS = len(EMOTIONS)
D = 8                              # feature dimension per modality
print('ready, emotions:', EMOTIONS)

## 2. A synthetic two-stream dataset

Each utterance has an *audio* feature vector and a *video* feature vector. Both depend on the true emotion, so either modality alone is informative, but neither is perfect. The class centers differ between the two streams, which is realistic: the voice and the face encode emotion differently.

In a real notebook this block is replaced by features extracted from an audiovisual Arabic affect set (a multimodal set if available; otherwise an Arabic speech-emotion set paired with visual features). The rest of the notebook stays the same.

In [ ]:
def make_centers(scale):
    # one center per class per modality
    return rng.normal(0, scale, size=(N_CLASS, D))

AUDIO_CENTERS = make_centers(1.6)
VIDEO_CENTERS = make_centers(1.4)

def sample_split(n_per_class, audio_noise=0.0):
    """Draw a labeled set; audio_noise is extra std added to the audio stream only."""
    Xa, Xv, y = [], [], []
    for k in range(N_CLASS):
        for _ in range(n_per_class):
            a = AUDIO_CENTERS[k] + rng.normal(0, 1.0 + audio_noise, size=D)
            v = VIDEO_CENTERS[k] + rng.normal(0, 1.0, size=D)
            Xa.append(a); Xv.append(v); y.append(k)
    return np.array(Xa), np.array(Xv), np.array(y)

# clean training data (audio and video both clean)
Xa_tr, Xv_tr, y_tr = sample_split(300)
print('train samples:', len(y_tr), '| audio dim:', Xa_tr.shape[1], '| video dim:', Xv_tr.shape[1])

## 3. A simple per-modality classifier

For each modality we use a nearest-centroid classifier: learn the mean feature vector of each class on the training data, then turn the (negative) distances to those means into class posteriors with a softmax. It is deliberately simple so the fusion effect, not the classifier, is what we see.

In [ ]:
def train_centroids(X, y):
    return np.array([X[y == k].mean(axis=0) for k in range(N_CLASS)])

def posteriors(X, centroids, temp=1.0):
    # negative squared distance to each class mean, softmaxed -> posterior per sample
    d2 = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)   # (n, N_CLASS)
    logits = -d2 / temp
    logits -= logits.max(axis=1, keepdims=True)
    p = np.exp(logits)
    return p / p.sum(axis=1, keepdims=True)

AUDIO_MODEL = train_centroids(Xa_tr, y_tr)
VIDEO_MODEL = train_centroids(Xv_tr, y_tr)
print('trained per-modality centroids:', AUDIO_MODEL.shape, VIDEO_MODEL.shape)

## 4. Late fusion and evaluation under noise

Late fusion combines the two streams at the decision level: we average the audio and video posteriors and take the argmax. We then sweep the test-time audio noise from clean to severe and record accuracy for three systems: audio-only, video-only, and the fused audiovisual model. The video stream is kept clean throughout, modeling a setting where the microphone degrades but the camera does not.

In [ ]:
def accuracy(pred, y): return float((pred == y).mean())

NOISE_LEVELS = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0]
rows = []
for nz in NOISE_LEVELS:
    Xa_te, Xv_te, y_te = sample_split(200, audio_noise=nz)
    pa = posteriors(Xa_te, AUDIO_MODEL)
    pv = posteriors(Xv_te, VIDEO_MODEL)
    pf = 0.5 * pa + 0.5 * pv                 # late fusion
    acc_a = accuracy(pa.argmax(1), y_te)
    acc_v = accuracy(pv.argmax(1), y_te)
    acc_f = accuracy(pf.argmax(1), y_te)
    rows.append((nz, acc_a, acc_v, acc_f))

print('chance level = %.2f' % (1.0 / N_CLASS))
print('%-8s %-12s %-12s %-14s' % ('noise', 'audio-only', 'video-only', 'audiovisual'))
for nz, a, v, f in rows:
    print('%-8.1f %-12.3f %-12.3f %-14.3f' % (nz, a, v, f))

## 5. The degradation curve

Plotting accuracy against audio noise shows the chapter's claim directly: the audio-only curve falls steeply toward chance, while the audiovisual curve stays high because the clean visual stream carries the decision once the audio is unreliable. The gap between the two curves is the measured value of the visual modality, and it is largest where the audio is worst.

In [ ]:
import matplotlib
matplotlib.use('Agg')   # headless; remove this line in a live notebook
import matplotlib.pyplot as plt

nz   = [r[0] for r in rows]
a_acc = [r[1] for r in rows]
v_acc = [r[2] for r in rows]
f_acc = [r[3] for r in rows]

plt.figure(figsize=(7, 4.5))
plt.plot(nz, a_acc, 'o-', label='audio-only')
plt.plot(nz, v_acc, 's--', label='video-only')
plt.plot(nz, f_acc, '^-', label='audiovisual (late fusion)')
plt.axhline(1.0 / N_CLASS, color='grey', ls=':', label='chance')
plt.xlabel('added audio noise (std)')
plt.ylabel('emotion accuracy')
plt.title('Video helps most when the audio is worst')
plt.legend(); plt.tight_layout()
plt.savefig('ch14_av_emotion_curve.png', dpi=120)
print('saved ch14_av_emotion_curve.png')
print('audio-only drop:  %.3f -> %.3f' % (a_acc[0], a_acc[-1]))
print('audiovisual drop: %.3f -> %.3f' % (f_acc[0], f_acc[-1]))

## 6. What this shows, and where real data plugs in

At zero noise, audio-only and audiovisual perform similarly: the extra modality adds little when the audio is already enough, which is exactly the chapter's caution that video does not always help. As the audio noise grows, audio-only accuracy falls toward chance while the audiovisual system stays high, because late fusion lets the clean video stream carry the decision. The size of the gap is the measured benefit of the visual modality, and reporting it as a curve over noise, against an audio-only baseline, is the honest way to evaluate any multimodal system.

To make this real for Arabic, replace Section 2 with features from an audiovisual Arabic affect set (a multimodal set if available; otherwise an Arabic speech-emotion set, such as the data used by ArabEmoNet, paired with visual features) and keep the rest. Two cautions carry over from the chapter. First, the audio and video must be synchronized: a small offset makes the visual stream misleading rather than helpful. Second, emotion labels are subjective, so report inter-annotator agreement, and because affective cues vary across dialects, evaluate per dialect rather than pooling. Late fusion is only one option; cross-attention fusion can do better on well-aligned data at higher cost.